# 📂 DQ Framework — Notebook 3: Dataset Loader

**Purpose:** Defines `load_dataset()` — loads a Spark DataFrame from
the source defined in a dataset config (Delta, Parquet, CSV, JSON, JDBC, view).

This notebook is `%run` by the controller. It has no local dependencies.

## Supported Source Types

In [0]:
SUPPORTED_SOURCE_TYPES = {"delta", "parquet", "csv", "json", "jdbc", "view"}

## DatasetLoaderError

In [0]:
class DatasetLoaderError(Exception):
    """Raised when a dataset cannot be loaded from its source."""
    pass

## load_dataset()

In [0]:
def load_dataset(spark, dataset_config):
    """
    Load a Spark DataFrame from the source defined in a dataset config dict.

    Parameters
    ----------
    spark          : active SparkSession (always available in Databricks notebooks)
    dataset_config : parsed dataset config dict (output of load_dataset_config)

    Returns
    -------
    pyspark.sql.DataFrame

    Raises
    ------
    DatasetLoaderError if the source cannot be read.

    Source config examples
    ----------------------
    # Delta table or view
    source:
      type: delta
      path: main.sales.orders            # catalog.schema.table

    # Parquet files on DBFS or cloud storage
    source:
      type: parquet
      path: /Volumes/main/landing/raw/orders/

    # CSV files
    source:
      type: csv
      path: /Volumes/main/landing/raw/orders/
      header: true
      inferSchema: true
      delimiter: ","

    # External view
    source:
      type: view
      path: my_catalog.my_schema.my_view

    # JDBC (e.g. Oracle, SQL Server)
    source:
      type: jdbc
      url:  jdbc:sqlserver://host:1433;database=mydb
      table: dbo.orders
      properties:
        user: myuser
        password: "{{secrets/scope/jdbc-password}}"
        driver: com.microsoft.sqlserver.jdbc.SQLServerDriver
    """
    ds_block = dataset_config["dataset"]
    name     = ds_block["name"]
    source   = ds_block.get("source", {})
    stype    = source.get("type", "delta").lower()
    path     = source.get("path", "")

    if stype not in SUPPORTED_SOURCE_TYPES:
        raise DatasetLoaderError(
            f"Dataset '{name}': unsupported source type '{stype}'.\n"
            f"  Allowed: {sorted(SUPPORTED_SOURCE_TYPES)}"
        )

    try:
        # ── Delta table or view ────────────────────────────────────────────────
        if stype in ("delta", "view"):
            return spark.table(path)

        # ── Parquet ───────────────────────────────────────────────────────────
        elif stype == "parquet":
            opts = source.get("options", {})
            return spark.read.options(**opts).parquet(path)

        # ── CSV ──────────────────────────────────────────────────────────────
        elif stype == "csv":
            opts = {
                "header":      str(source.get("header",      True)).lower(),
                "inferSchema": str(source.get("inferSchema", True)).lower(),
                "delimiter":   source.get("delimiter", ","),
            }
            opts.update(source.get("options", {}))
            return spark.read.options(**opts).csv(path)

        # ── JSON ──────────────────────────────────────────────────────────────
        elif stype == "json":
            opts = source.get("options", {})
            return spark.read.options(**opts).json(path)

        # ── JDBC ──────────────────────────────────────────────────────────────
        elif stype == "jdbc":
            url        = source.get("url", "")
            table      = source.get("table", path)
            properties = source.get("properties", {})
            if not url:
                raise DatasetLoaderError(
                    f"Dataset '{name}': JDBC source requires 'url' in the source config."
                )
            return spark.read.jdbc(url=url, table=table, properties=properties)

    except DatasetLoaderError:
        raise
    except Exception as exc:
        raise DatasetLoaderError(
            f"Dataset '{name}': failed to load from {stype} source '{path}'.\n"
            f"  Error: {exc}"
        ) from exc

### ✅ Dataset loader notebook loaded

Defines: `DatasetLoaderError`, `load_dataset()`, `SUPPORTED_SOURCE_TYPES`